## Quantization in Large Language Models

### **Introduction to Quantization**

Quantization is a process used to **reduce the memory requirements** and **computational complexity** of large machine learning models. By representing model parameters with lower-precision values, quantization makes it possible to run models more efficiently on devices with **limited memory and computational resources**.

For large language models (LLMs), quantization can:
- **Reduce Memory Usage:** Lower-precision data types (such as int8) use less memory than higher-precision types (like float32), allowing models to fit into memory-constrained environments.
- **Improve Inference Speed:** By using simpler operations on smaller data types, quantization can reduce the time it takes for a model to process inputs and generate outputs.
- **Preserve Accuracy:** Quantization is carefully designed to minimize the impact on model accuracy, though a trade-off often exists between precision and efficiency.

We will focus on Post-Training Quantization (PTQ), a quantization technique that applies quantization to a **pre-trained model**. PTQ is a popular method for quantizing large language models because it can be applied to a **wide range of models** that we may want to use in **inference mode**. By contrast, **quantization-aware training** (QAT) requires retraining the model with quantization in mind, which can be **more complex and time-consuming.**

We will first get a general understanding of quantization by manually implementing two commonly adopted approaches: **absmax and minmax (or zero-point). **

Next, we will explore two different ways (one using PyTorch, the other using HuggingFace) to run a **dynamic quantization** (i.e., PTQ where only the weights are quantized, and not the activations).

**Absmax** (Absolute Maximum) scaling is the simplest method. It centers the data around zero and scales everything based on the single largest absolute value in the tensor.
 It finds the maximum absolute value $|x|_{max}$ and maps it to the range of an 8-bit integer ($-127$ to $127$).

 **minmax** This method is more precise because it "shifts" the data to fit perfectly into the $0$ to $255$ range (for $UINT8$). This is where the Zero-point comes in.
 Instead of centering on zero, it finds the actual min and max of your data. It calculates a Scale and a Zero-point (an integer offset that represents what "zero" was in the original data).



## 1. Absmax and Minmax Quantization

The goal of quantization is, remember, mapping continuos values (e.g., float32) into a discrete set of values (e.g., int8).

So let's create a matrix $W$ and a vector $x$ to be quantized. Let's initialize them randomly (but, just to see what happens, let's set W[0,0] and x[0] = 0).

In [60]:
import torch

torch.random.manual_seed(0)

n_rows = 3
n_cols = 5

# TODO: Create two random tensors, W (of shape n_rows, n_cols) and x (of shape n_cols)
W = torch.rand(n_rows, n_cols)
x = torch.rand(n_cols)


W[0,0] = 0
x[0] = 0
# to see how "zero" is handled in both methods.

# In Absmax, we only care about the largest absolute value.

In [61]:
print(W)
print(x)

tensor([[0.0000, 0.7682, 0.0885, 0.1320, 0.3074],
        [0.6341, 0.4901, 0.8964, 0.4556, 0.6323],
        [0.3489, 0.4017, 0.0223, 0.1689, 0.2939]])
tensor([0.0000, 0.6977, 0.8000, 0.1610, 0.2823])


Let's first compute the matrix multiplication to observe the result. This is the operation that we typically want to execute, and that we want to quantize.

We will quantize $W$ and $x$ separately, and then multiply them together. Finally, we will need to dequantize the result to compare it with the original result.

In [62]:
out = W @ x
print(out)

tensor([0.7148, 1.3109, 0.4083])


## 1.1 Absmax quantization

In absmax quantization, we use a symmetric range around 0. This means that we need to identify the maximum absolute value in the matrix $W$ and the vector $x$.

We define a function `absmax_quantize` that takes as input any tensor and produces a version of the same tensor, but quantized.

In [64]:
def absmax_quantize(W):
    # NOTE: we assume that we always map to 8-bit integers

    # TODO: find the scale factor that maps the maximum absolute value in W to the maximum value of int8

    max_value = W.abs().max()
    # 1. Find the absolute max

    scale = 127 / max_value # how "long" the step between any two int8 values is
    # 2. Calculate scale

    # TODO: quantize W using the scale factor (hint: remember to round to the nearest integer and convert to int8)
    W_q = (W * scale).round().to(torch.int8)
    # 3. Quantize

    return W_q, scale


# In Absmax, we only care about the largest absolute value.
# Let’s say the largest number in your W tensor is 0.95. 1
# The Scale: All numbers are divided by 0.95 and multiplied by 127. 2
# Zero Handling: Since there is no "shift," the 0 you manually set at W[0,0] will always remain 0 in the quantized version.
# Symmetry: It assumes the range is roughly [-0.95, 0.95].
# If your tensor only has positive numbers (like torch.rand produces), you are wasting half of your 8-bit range (the negative side).

Notice, we return both the quantized tensor and the scale factor. **The scale factor is used to dequantize the tenso**r. So we might as well define a dequantize function:

In [65]:
# Absmax dequantization is the process of reversing the quantization to get your numbers back from 8-bit integers (INT8)
# to floating-point values (FP32).
# Since Absmax is symmetric, the formula is very straightforward.
# You simply take your quantized integer and divide it by the same scale factor you used to shrink it.

def absmax_dequantize(W_q, scale):
    # TODO: dequantize W_q using the scale factor
    dequantized_W = W_q.float() / scale
    return dequantized_W

Let's get the quantized version of W, and of x. Then, we can check how much we are losing by quantizing the values.

In [66]:
W_q, scale_W = absmax_quantize(W)
x_q, scale_x = absmax_quantize(x)

In [67]:
print("W: " ,W)
print("W_q: ",W_q)


#TODO: dequantize W_q and check how close it is to the original W
W_deq = absmax_dequantize(W_q, scale_W)
print("W_deq: ",W_deq)
W_diff = (W - W_deq)
print("dif: ", W_diff)
mean = W_diff.abs().mean()
print(mean)

W:  tensor([[0.0000, 0.7682, 0.0885, 0.1320, 0.3074],
        [0.6341, 0.4901, 0.8964, 0.4556, 0.6323],
        [0.3489, 0.4017, 0.0223, 0.1689, 0.2939]])
W_q:  tensor([[  0, 109,  13,  19,  44],
        [ 90,  69, 127,  65,  90],
        [ 49,  57,   3,  24,  42]], dtype=torch.int8)
W_deq:  tensor([[0.0000, 0.7694, 0.0918, 0.1341, 0.3106],
        [0.6353, 0.4870, 0.8964, 0.4588, 0.6353],
        [0.3459, 0.4023, 0.0212, 0.1694, 0.2965]])
dif:  tensor([[ 0.0000e+00, -1.1678e-03, -3.2846e-03, -2.0833e-03, -3.1565e-03],
        [-1.1972e-03,  3.0486e-03, -5.9605e-08, -3.1824e-03, -2.9696e-03],
        [ 3.0211e-03, -6.2406e-04,  1.1499e-03, -5.4795e-04, -2.5736e-03]])
tensor(0.0019)


In [68]:
print("x: ", x)
print("x_q: ",x_q)


# TODO: dequantize x_q and check how close it is to the original x
x_deq = absmax_dequantize(x_q, scale_x)
print("x_deq: ",x_deq)
x_diff = (x - x_q)
print("dif: ",x_diff)
mean = x_diff.abs().mean()
print(mean)

# abs() positive

x:  tensor([0.0000, 0.6977, 0.8000, 0.1610, 0.2823])
x_q:  tensor([  0, 111, 127,  26,  45], dtype=torch.int8)
x_deq:  tensor([0.0000, 0.6992, 0.8000, 0.1638, 0.2835])
dif:  tensor([   0.0000, -110.3023, -126.2000,  -25.8390,  -44.7177])
tensor(61.4118)


Notice that, in both cases, absmax maps the value 0 to 0. This is a good property, as it allows us to represent the zero value without losing any information. This property stems from the symmetry around 0 we imposed.

However, do note that we are also "wasting" some bits of the range! Can you spot where?

Let's now compute the matrix multiplication between W_q and x_q.

In [69]:
# TODO: perform the matrix multiplication using the quantized values
product = W_q @ x_q
print(product)

tensor([96, 88,  6], dtype=torch.int8)


In [70]:
print(out)
print(out.dtype)

tensor([0.7148, 1.3109, 0.4083])
torch.float32


Can you see that there's something wrong? Let's see what one of the rows of W_q and x_q contain:

In [71]:
W_q[0], x_q

# The range of an int8 is only -128 to 127. An int8 bucket is tiny. It can only hold a value up to 127.
# If W_q and x_q are both int8, their product can be as large as 127 * 127 = 16,129.
# That already requires int16. Once you start adding those products together (accumulation),
# you can easily cross the 32,767 limit of int16, pushing you into int32 territory.


# in math problems: We store in 8-bit, but we calculate in 32-bit

# the "spillage" (overflow) corrupted the data.
# 109 * 111 = 12,099
# 12099 is bigger than 127

(tensor([  0, 109,  13,  19,  44], dtype=torch.int8),
 tensor([  0, 111, 127,  26,  45], dtype=torch.int8))

The dot product of these two vectors definitely isn't what we get as the first number of the matrix multiplication -- i.e. (W_q @ x_q)[0]. Indeed, we can run as int16, and see that the result is quite different:

In [72]:
# TODO: perform the matrix multiplication using the quantized values converted to int16
out_q = W_q.to(torch.int16) @ x_q.to(torch.int16)
print(out_q)

tensor([16224, 29528,  9222], dtype=torch.int16)


The result of the dot product overflows the int8 range. This is a well-known problem. Indeed, the accumulation of results, in quantization, is typically done with higher precision than the single values. This is tricky to do in pure Python/PyTorch, but can be done efficiently in other ways.

Let's stick to the simple approach for now.

To get the correct result, we need to dequantize the result. This is done by multiplying the result by the scale factor of the two operands.

In [73]:
# TODO: dequantize the result and check how close it is to the original result
out_deq = absmax_dequantize(absmax_dequantize(out_q, scale_W), scale_x)

# out_deq = out_q / scale_W / scale_x
# out_deq = absmax_dequantize(out_q, scale_W * scale_x)

print(out_deq)
print(out)

# When you multiplied the two quantized numbers, you essentially created a "double-scaled" number.
# To get back to the original tiny decimal, you have to peel back both layers of "packing."
# Divide by Scale A, then divide by Scale B.

# Before: Your original result was 0.7148 (a tiny float).
# Middle: Your quantized product was 96 (a big integer).
# After: If you take 96 and divide it by both scales, you should land right back near 0.7148.

tensor([0.7214, 1.3129, 0.4101])
tensor([0.7148, 1.3109, 0.4083])


Remember, our goal was `out`. How much did we lose by quantizing and dequantizing?

In [74]:
#TODO: check the difference between the original and the dequantized result
out_diff = out - out_deq
print(out_diff)
print(out_diff.abs().mean())

tensor([-0.0066, -0.0020, -0.0018])
tensor(0.0035)


## 2. Minmax Quantization

In minmax quantization, we use the **minimum and maximum** values in the **matrix** $W$ and the **vector** $x$ to define the **range**. In this way, **we get a range that is as tight as possible around the values we are quantizing**. This will, however, **change the zero value, which will not be mapped to 0 anymore**.



In [78]:
# minmax is smarter than the "Absmax" method because it doesn't assume your numbers are centered around zero
# minmax "stretches" and "shifts" the numbers to use every bit available in the int8 range.
# Scaling (scale): Making the "window" of your data fit into the size of an 8-bit integer.
# Shifting (zero_point): Moving the data left or right so that the lowest value in your weights becomes -128 and the highest becomes 127.
# zero_point: shifting the data, "0.0" in the real world is no longer "0" in the integer world.
# In Min-Max, 0.0 might equal something like -12 or +45.
# The zero_point tells the computer exactly where the original "zero" is hiding inside the quantized integers.

def minmax_quantize(W):
    # the following notations come from:
    # (1) scaling W to [0,1] ==> W' = (W - min(W)) / (max(W) - min(W)),
    # (2) scaling W' to [-128, 127] ==> W_q = W' * 255 - 128
    # by combining the two, we get that:
    # W_q = W * scale + offset
    # we will call the offset "zero_point", as it represent the value that maps to 0

    # TODO: find the scale factor that maps the minimum and maximum values in W to -128 and 127
    min = W.min()
    max = W.max()
    delta = max - min # the range of values in W
    scale = 255 / delta # how "long" the step between any two int8 values is
    zero_point = -(128 * max + 127 * min) / delta # the value that maps to 0

    # TODO: quantize W using the scale factor (hint: remember to round to the nearest integer and convert to int8)
    W_q = (W * scale + zero_point).round().to(torch.int8)

    return W_q, scale, zero_point

# Since we added the zero_point and multiplied by the scale to quantize, we have to do the exact opposite to go back.

def minmax_dequantize(W_q, scale, zero_point):
    # TODO: dequantize W_q using the scale factor and zero_point
    dequantized_W = W_q.float() - zero_point # unshift
    dequantized_W = dequantized_W / scale    # unshrink
    return dequantized_W


# Absmax would map these to positive integers ($0$ to $127$) and leave the negative side ($-128$ to $0$) completely empty.
# You waste half your memory's potential!
# Min-Max would shift the data so $0.5$ becomes $-128$ and $0.8$ becomes $127$. You use the full range,
# which makes the quantization much more accurate.

In [79]:
W_q, scale_W, zero_point_W = minmax_quantize(W)
x_q, scale_x, zero_point_x = minmax_quantize(x)

Let's see the results for $W$ (same considerations will apply for $x$).

In [80]:
print(W_q)
print(W)

#TODO: dequantize W_q and check how close it is to the original W
W_deq = minmax_dequantize(W_q, scale_W, zero_point_W)
print("W_deq: ",W_deq)
W_diff =  W - W_deq
print("dif: ", W_diff)
mean = W_diff.abs().mean()
print(mean)
print("zero point", zero_point_W)

tensor([[-128,   91, -103,  -90,  -41],
        [  52,   11,  127,    2,   52],
        [ -29,  -14, -122,  -80,  -44]], dtype=torch.int8)
tensor([[0.0000, 0.7682, 0.0885, 0.1320, 0.3074],
        [0.6341, 0.4901, 0.8964, 0.4556, 0.6323],
        [0.3489, 0.4017, 0.0223, 0.1689, 0.2939]])
W_deq:  tensor([[0.0000, 0.7699, 0.0879, 0.1336, 0.3058],
        [0.6328, 0.4887, 0.8964, 0.4570, 0.6328],
        [0.3480, 0.4008, 0.0211, 0.1687, 0.2953]])
dif:  tensor([[ 0.0000e+00, -1.6660e-03,  5.9070e-04, -1.5574e-03,  1.5770e-03],
        [ 1.2942e-03,  1.4431e-03,  5.9605e-08, -1.3830e-03, -4.7821e-04],
        [ 8.6200e-04,  9.5379e-04,  1.2329e-03,  1.1641e-04, -1.4110e-03]])
tensor(0.0010)
zero point tensor(-128.)


First, notice that 0 no longer maps to 0! Indeed, it maps to zero_point_W (after rounding). This implies that the dequantization of 0 will no longer be 0. This may be a problem!

But, notice that we are using the **full range of the int8** values. This means that we are **not wasting any bits of the range**! (the minimum value is -128, the maximum value is 127). This can also be seen in the average absolute error, which is lower than what we had with absmax.

Similarly to what we did before, let's compute the output of the operation, and then dequantize it!

As you noticed, 0 no longer maps to 0.
In neural networks, we often have "padding" or "dropout" where 0 is used to mean "ignore this." If 0 becomes -12 or some other number after quantization, the model might start "hallucinating" or paying attention to things it should ignore.

Feature        |  Absmax (Symmetric)          |  Min-Max (Asymmetric)
Zero Mapping   |  0.0 is exactly 0            |  0.0 is a "Zero Point"
integerFormula |  X_q = X . S                 |  X_q = X . S + Z
Dot Product    |  Simple: W_q x_q             |  Complex: W_q x_q + cross terms          |  AccuracyLower (wastes bits) |  Higher (uses full int8 range)

In [84]:
#TODO:  perform the matrix multiplication using the quantized values converted to int16
out_q = W_q.to(torch.int16) @ x_q.to(torch.int16)
print(out_q)

tensor([20345,  8377, -5266], dtype=torch.int16)


The dequantification is a bit trickier, in this case. Can you figure out why we need the following operations?

Hint: consider the transformation we are applying to each value (value * scale + zero_point). What happens when we compute the dot product?

In [82]:
out_deq = (out_q - W.shape[1] * zero_point_W * zero_point_x - W.sum(axis=1) * scale_W * zero_point_x - x.sum() * scale_x * zero_point_W) / (scale_W * scale_x)

In [83]:
print(out_deq)
print(out)
print((out_deq - out).abs().mean())

tensor([0.7148, 1.3106, 0.4080])
tensor([0.7148, 1.3109, 0.4083])
tensor(0.0002)


<span style="color:red">Extra stuff!</span>

We could have computed scales and zero points at different granularities (e.g., for each row, or column of $W$). How would that have changed the results? What changes would we have to do to the code?

# Dynamic quantization

In this second part, we will apply **dynamic quantization** by using **PyTorch or HuggingFace** (with BitsAndBytes). We will quantize both to **8** and to **4** bits, and we will see how that **affects LLMs** (in terms of memory and speed).

In [93]:
import torch
import os
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

In [95]:
from huggingface_hub import login

# TODO: Login to the Hugging Face model hub to be able to upload models
with open("/content/token.txt", "r") as f:
    token = f.read()
    f.close()

login(token=token)

First, let's load our model (Llama 3.2 1B) and let's see some base statistics (memory usage, inference time).

In [104]:
model_id = "meta-llama/Llama-3.2-1B"
#TODO: load the model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [105]:
def get_model_size(model):
    """Get the size of the model in MB"""
    torch.save(model.state_dict(), "temp.pth")
    # the state_dict, which contains all the weights and biases
    size = os.path.getsize("temp.pth") / 1e6  # size in "MB" (technically, it should be 1024**2, but we approximate to 1e6 to get an easier conversion #params <=> MB)
    # It asks the operating system, "How many bytes is this file?"
    os.remove("temp.pth")
    # It cleans up the temporary file so you don't clutter your computer.
    return size

print(f"Model size before quantization {(get_model_size(model)):.2f} MB")

Model size before quantization 2471.68 MB


Wait, wasn't Llama 1B supposed to be 4GB (4 bytes * 1B parameters)? Why do we get ~ 5 GB (i.e., 1.25B parameters)?

We are not considering the parameters used in the embedding layer (you can count how many parameters you have in the embedding layer and see that it matches the difference).

Additionally, the count does not include the `lm_head`, i.e. the layer used to go from the hidden states to the logits. This is because in Llama (and other models) the `lm_head` is shared with the embedding layer, so it is not counted twice.

In [106]:
text = "The secret of life is"
# Notice we use a batch of 20 sentences -- we will get better results
# on quantized models when processing a batch of inputs
text = [text]*20

#TODO: encode the text using the tokenizer
inputs = tokenizer(text, return_tensors="pt", padding=True)

tic = time.time()

with torch.no_grad():
    #TODO: generate the output from the model
    baseline_output = model.generate(**inputs, max_new_tokens=100)

elapsed_time = time.time() - tic

#TODO: decode the output
baseline_decoded = tokenizer.batch_decode(baseline_output[0] , skip_special_tokens=True)

print("Baseline model output:", baseline_decoded)
print("\nTime taken for baseline model:", elapsed_time)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Baseline model output: ['The secret of life is in the details. The details are the things that make the difference. The details are the things that matter. The details are the things that make the difference.\nThe details are the things that matter.\nThe details are the things that make the difference.\nThe details are the things that matter. The details are the things that make the difference. The details are the things that matter. The details are the things that make the difference.\nThe details are the things that matter. The details are the things that']

Time taken for baseline model: 600.912024974823


Dynamic quantization a**pplies lower precision to model weights** and activations at runtime. This method doesn’t require modifications to the model architecture or retraining, which makes it relatively easy to apply.

- **Advantages:**
  - Quick to implement with **minimal changes**.** No calibration** step is needed.

- **Limitations:**
  - Activations are **not** **pre-quantized** **bold text**, meaning some precision is maintained but at the cost of slightly higher resource use at inference time.

We can use the `quantize_dynamic()` function, available in PyTorch, to apply dynamic quantization to a model.

We can specify a set of layer types to be quantize. Let's stick with Linear layers. We specify the desired type (represented by torch.qint8) , and off we go!

In [101]:
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    # Only quantize the Linear layers (the dense layers)
    # In Transformers (like Llama), the vast majority of the parameters live in these Linear layers (the Attention and MLP projections).
    # We usually leave layers like LayerNorm or the Embedding in floating-point because they are sensitive; quantizing them can often "break" the model's logic.
    dtype=torch.qint8
).to('cpu')

# Dynamic means that the weights are converted to int8 now (static), but the activations (the data passing through the model) are converted to int8 later, during the actual prediction.
# This is the easiest form of quantization because you don't need to "calibrate" the model with a specific dataset first.

# Model size after quantization
print(f"Model size after quantization {(get_model_size(quantized_model)):.2f} MB")

/tmp/ipykernel_1445/2062498388.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Model size after quantization 1761.35 MB


We use **[TorchAO](https://docs.pytorch.org/ao/stable/api_ref_quantization.html?utm_source=chatgpt.com)** to apply dynamic quantization to a model. TorchAO is the new quantization framework that replaces the deprecated `torch.ao.quantization` APIs.  

We can specify a set of layer types to be quantize. Let's stick with Linear layers. We specify the desired type, and off we go!

TorchAO (Architecture Optimization) is the newer, high-performance way to quantize models, specifically designed to be more flexible and faster.

While the old quantize_dynamic is a bit of a "black box," torchao allows for more granular control over how the math is performed.

While the old quantize_dynamic is a bit of a "black box," torchao allows for more granular control over how the math is performed.

In [107]:
import torch
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig

quantize_(
    model,
    Int8DynamicActivationInt8WeightConfig(),
    # This is the heart of the operation. It tells the model to use 8-bit integers for two things:
    # Weights: The stored values (the "knowledge" of the model).
    # Dynamic Activations: The data flowing through the layers.
    # This is the most common form of "post-training quantization" because it balances memory savings with accuracy.
    filter_fn=lambda m, name: isinstance(m, torch.nn.Linear),
    # gatekeeper just linear layers
    device="cpu",
)

print(f"Model size after quantization {get_model_size(model):.2f} MB")

Model size after quantization 1762.34 MB


Okay -- 2.3GB? Why not 5GB / 4 = 1.25GB? After all, we are going from float32 to int8.

That's correct -- technically. Except, we are only encoding linear layers, and not the embedding layer. That means that, of the original 1.25B parameters, we are only quantizing 1B. The rest, in the embedding layer, is kept as float32.

If you run the numbers, though, you should still find a problem: 1B * 1 byte + 0.25B * 4 bytes = 2GB. What about the rest? There's one more thing: remember, the `lm_head` was shared with the Embedding layer. However, since it is "copied" into a linear layer in Llama, the quantization process will quantize it as well. So that's an extra 0.25B parameters encoded as int8 -- hence 2.3GB.

Finally, we could technically also quantize the embeddings (it has been introduced in later versions of PyTorch), but for simplicity we will not do it here (it would require some additional steps).

In [108]:
quantized_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): DynamicQuantizedLinear(in_features=2048, out_features=2048, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
          (k_proj): DynamicQuantizedLinear(in_features=2048, out_features=512, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
          (v_proj): DynamicQuantizedLinear(in_features=2048, out_features=512, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
          (o_proj): DynamicQuantizedLinear(in_features=2048, out_features=2048, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
        )
        (mlp): LlamaMLP(
          (gate_proj): DynamicQuantizedLinear(in_features=2048, out_features=8192, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
          (up_proj): DynamicQuantizedLinear(in_features=2048, out_features=8192, dtype=torch.qint8, qscheme=torch.per_

In [109]:
tic = time.time()

with torch.no_grad():
    #TODO: generate the output from the quantized model
    output = model.generate(**inputs, max_new_tokens=100)

elapsed_time = time.time() - tic

#TODO: decode the output
output_decoded = tokenizer.batch_decode(output[0], skip_special_tokens=True)

print("Quantized model output:", output_decoded)
print("\nTime taken for baseline model:", elapsed_time)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Quantized model output: ['The secret of life is to love and be loved\nThe secret of life is to love and be loved\nThe secret of life is to love and be loved\nThe secret of life is to love and be loved\nThe secret of life is to love and be loved\nThe secret of life is to love and be loved\nThe secret of life is to love and be loved\nThe secret of life is to love and be loved\nThe secret of life is to love and be loved\nThe secret of life is to']

Time taken for baseline model: 2512.1927053928375


Hugging Face provides several built-in quantization options, each suited to different model and deployment needs:
https://huggingface.co/docs/transformers/v4.46.0/quantization/overview

For this lab, we will use `Quanto`.

In [115]:
pip install optimum-quanto

In [117]:
pip install quanto accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 2.5 MB/s eta 0:00:00


In [118]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig
import torch

model_id = "meta-llama/Llama-3.2-1B"

# Quantize to 8-bit weights
quant = QuantoConfig(weights="int8")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant,
    device_map="cpu",
)

print(f"Model size after quantization: {get_model_size(model)} MB")

ImportError: Loading an optimum-quanto quantized model requires optimum-quanto library (`pip install optimum-quanto`)

In [ ]:
tic = time.time()

with torch.no_grad():
    #TODO: generate the output from the quantized model
    output = ...

elapsed_time = time.time() - tic

#TODO: decode the output
output_decoded = ...

print("\nquantized model output:", output_decoded)
print("\nTime taken for baseline model:", elapsed_time)